### dataset 65
#### Johanna Tilešová - 50%
#### Adam Pečenka - 50%

# Fáza 1: Prieskum dát (Exploratory Data Analysis)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import scipy.stats as stats
import numpy as np
from statsmodels.stats.power import TTestIndPower


In [ ]:
mpl.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["font.family"] = "DejaVu Sans"
sns.set_theme(palette="vanimo")

### Načítanie dát do premenných

In [ ]:
df_observation = pd.read_csv('datasets/observation.csv',sep='\t')
df_patient = pd.read_csv('datasets/patient.csv',sep='\t')
df_station = pd.read_csv('datasets/station.csv',sep='\t')

# 1.1 Základný opis dát spolu s ich charakteristikami

Pre kazdy z datasetov sme vykonali nasledujúce kroky:
#### 1. Prieskum dát - základné štatistiky a informácie o dátach
1. `info()`
2. `isnul().sum()`
3. `head()`
4. `nunique()` - len unique hodnoty
5. `describe()` - štatistiky

#### 2. Analýza atribútov
1. Typy atribútov - `select_dtypes()`
    - `select_dtypes().columns`
    - `select_dtypes().nunique()`

#### 3. Vysledok analýzy - poznámky o dátach, návrhy na úpravy


## Dataset Observation

### 1. Prieskum dát

In [ ]:
df_observation.shape

In [ ]:
df_observation.info()

In [ ]:
df_observation.isnull().sum()

In [ ]:
df_observation.head()

In [ ]:
df_observation.nunique()

In [ ]:
df_observation.describe()

### 2. Analýza atribútov

In [ ]:
len(df_observation.select_dtypes(include=['float64']).columns)

In [ ]:
df_observation.select_dtypes(include=['float64']).columns

In [ ]:
df_observation.select_dtypes(include=['float64']).nunique()

In [ ]:
df_observation['oximetry'].value_counts().sort_index()

### 3. Vysledok analýzy
- dataset pozostáva z 23 atribútov a 12047 záznamov
- všetky atribúty sú číselné typu `float64` - ide o kontinuálne merania z rôznych senzorov
- neobsahuje žiadne chýbajúce hodnoty

#### Oximetry
- `1.0` → Oxymeter dokázal získať validný SpO₂ signál
- `0.0` → Oxymeter nedokázal získať signál / meranie zlyhalo

=> predikovou premennou nie je „koľko SpO₂“, ale „či sa SpO₂ dalo zmerať“



## Dataset Patient

### 1. Prieskum dát

In [ ]:
df_patient.shape

In [ ]:
df_patient.info()

In [ ]:
df_patient.isnull().sum()

In [ ]:
df_patient.head()

In [ ]:
df_patient.nunique()

In [ ]:
df_patient.describe()

In [ ]:
df_patient.groupby('user_id').head()

### 2. Analýza atribútov

In [ ]:
print('Pocet atributov typu int a float:',  len(df_patient.select_dtypes(include=['int64','float64']).columns))

In [ ]:
df_patient.select_dtypes(include=['int64','float64']).columns

In [ ]:
df_patient.select_dtypes(include=['int64','float64']).nunique()

In [ ]:
print('Pocet atributov typu object:', len(df_patient.select_dtypes(include=['object']).columns))

In [ ]:
df_patient.select_dtypes(include=['object']).columns

In [ ]:
df_patient.select_dtypes(include=['object']).nunique()

### 3. Vysledok analýzy
- dátumy ako `birth_date` a `registration` sú ako object --> mali by sa konvertovať na datetime
- `current_location` je ako object --> mali by sa rozložiť na dve číselné premenné (latitude, longitude) typu `float64`
- atribút `residence` má veľa hodnot typu NaN

## Dataset Station
### 1. Prieskum dát

In [ ]:
df_station.shape

In [ ]:
df_station.info()

In [ ]:
df_station.isnull().sum()

In [ ]:
df_station.head()

In [ ]:
df_station.nunique()

In [ ]:
df_station.describe()

### 2. Analýza atribútov

In [ ]:
print(f"Pocet atributov typu float: {len(df_station.select_dtypes(include=['float64']).columns)}")

In [ ]:
df_station.select_dtypes(include=['float64']).nunique()

In [ ]:
print(f"Pocet atributov typu object: {len(df_station.select_dtypes(include=['object']).columns)}")

In [ ]:
df_station.select_dtypes(include=['object']).nunique()

### 3. Vysledok analýzy
Dataset `station` predstavuje miesta o senzorových staniciach, kde sa meria oximetria
Vzťah k iným tabuľkám => pacient sa nachádza na nejakej stanici alebo jeho údaje pochádzajú z konkrétneho zariadenia

- vacsina atribútov je typu `object` - názvy staníc, lokality, typy zariadení
- atribúty `latitude` a `longitude` sú číselné typu `float`

Potenciálne problémy:
  - nejednotný spôsob označovania staníc (text vs kód)
  - chýbajúce údaje pri lokalite
  - duplicitné záznamy (viac názvov pre tú istú stanicu)


## Analyza jednotlivych atribútov
Cielom je zistiť, ktoré atribúty najviac súvisia s tým, či je oximetry = 0 alebo 1

### Popis parametrov senzorov

In [ ]:
sensor = pd.read_csv('sensor_variable_range.csv',sep='\t')
sensor

### Validácia rozsahov atribútov v dátach

In [ ]:
df_observation.describe().T

Atribút `oximetry` (predikovaná premenná) má min 0.0, čo je fyziologicky nemožné.

## Vztahy medzi atribútmi

In [ ]:
corr_target = df_observation.corr()['oximetry'].sort_values(ascending=False)
corr_target

In [ ]:
mpl.rcParams["font.family"] = "DejaVu Sans"

# vypočítame koreláciu celej matice
corr = df_observation.corr()

# zoradíme podľa korelácie s targetom
corr_sorted = corr.sort_values(by='oximetry', ascending=False)

plt.figure(figsize=(8, 8))
sns.heatmap(
    corr_sorted[['oximetry']],        # len stĺpec targetu
    annot=True,
    cmap="PiYG",
    center=0,
    vmin=-1, vmax=1
)
plt.title("Korelácia všetkých atribútov s oximetry (globálne porovnanie)")
plt.tight_layout()
plt.show()

Graf potvrdzuje, že z celého datasetu majú najvyššiu koreláciu s premennou `oximetry` práve signálové atribúty (Motion/Activity index, PI, Signal Quality Index, EtCO₂). Ostatné fyziologické premenné (HR, BP, CO, SpO₂) majú takmer nulovú väzbu, čo podporuje rozhodnutie zamerať sa pri ďalšom spracovaní na signálové a nie vitálne atribúty.

Z korelácie vyplýva, že úspešnosť oximetrického merania (`oximetry`) súvisí
najmä so stavom signálu – najvyššia korelácia sa objavila pri `Motion/Activity index`
(pohyb spôsobuje artefakty) a `PVI` (kvalita periférnej perfúzie). Fyziologické
parametre ako HR, BP alebo SpO₂ s `oximetry` takmer vôbec nekorelujú, čo potvrdzuje,
že problém nevzniká z nedostatku kyslíka, ale z technických/meracích podmienok.

Signal Quality Index (SQI) sme zaradili medzi kľúčové atribúty namiesto RR alebo FiO₂,
pretože SQI priamo odráža technickú dostupnosť PPG signálu. Ak je SQI nízke, senzor
nedokáže zmerať saturáciu bez ohľadu na fyziologický stav pacienta. 

RR a FiO₂ iba
opisujú respiračný stav pacienta, nie kvalitu snímania, a preto majú na `oximetry`
len nepriamy alebo minimálny vplyv.



### Vybranych 10 významnych atribútov
| Atribút               | Co to znamená                                 |
|-----------------------|-----------------------------------------------|
| PI (Perfusion Index)  | kvalita signálu                    |
| Motion/Activity index | artefakty z pohybu                            |
| Signal Quality Index  | kvalita merania                               |
| Skin Temperature      | periférna perfúzia ovplyvňuje PPG             |
| EtCO₂                 | stabilita dýchania                            |
| RR                    | respiračná stabilita                          |
| PRV                   | autonómny stav                                |
| BP                    | perfúzia                                      |
| SV                    |                     |
| FiO₂                  | nie priamy, ale prostredníctvom stavu pacienta |


In [ ]:
top10Attributes = ["PI", "Motion/Activity index", "Signal Quality Index", "Skin Temperature",
         "EtCO₂", "RR", "PRV", "BP", "SV", "FiO₂"]

fig, axes = plt.subplots(5, 2, figsize=(16, 18))
axes = axes.flatten()

for ax, col in zip(axes, top10Attributes):
    sns.histplot(df_observation[col], kde=True, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
REF_TOP10 = {
    "PI": (0.2, 20),                    # Perfusion Index – kľúčový, priamo ovplyvňuje možnosť merania
    "Motion/Activity index": (None, None),  # pohyb -> artefakty, častá príčina oximetry=0
    "Signal Quality Index": (0, 100),   # hodnotí úspešnosť snímania
    "Skin Temperature": (33, 38),       # nízka teplota -> vazokonstrikcia -> slabý PPG signál
    "EtCO₂": (35, 45),                  # stabilita dychu, sekundárny indikátor
    "RR": (12, 20),                     # stabilita ventilácie
    "PRV": (20, 200),                   # autonómny stav, môže indikovať stres/kolísanie
    "BP": (90, 120),                    # perfúzia na systémovej úrovni
    "SV": (60, 100),                      
    "FiO₂": (21, 100),                  # podporný parameter, nie priamy, ale indikuje stav pacienta
}

_Poznámka: pre `Motion/Activity index` nie je v meta-dátach uvedený rozsah (`Value Range = NaN`). Reálny rozsah preto bude potrebné odvodiť priamo z pozorovaných
hodnôt v datasete (napr. pomocou describe() / percentilov) v kroku 1.2 (cleaning)._

In [ ]:
def attribute_report(df, ranges):
    rows = []
    for col, (low, high) in ranges.items():
        s = df[col]
        n = len(s)
        below = (s < low).sum()
        above = (s > high).sum()
        rows.append({
            "attribute": col,
            "min": s.min(),
            "max": s.max(),
            "mean": s.mean(),
            "ref_low": low,
            "ref_high": high,
            "% out_of_range": round((below + above) / n * 100, 3),
        })
    return pd.DataFrame(rows)


In [ ]:
report_top10 = attribute_report(df_observation, REF_TOP10)
report_top10

#### Vzťahy medzi top 10 atribútmi

In [ ]:
mpl.rcParams["font.family"] = "DejaVu Sans"

top10_cols = list(REF_TOP10.keys())

corr = df_observation[top10_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="PiYG", square=True)
plt.title("Korelácie medzi TOP 10 atribútmi")
plt.show()

## Párová Analyza

In [ ]:
mpl.rcParams["font.family"] = "DejaVu Sans"

pair_vars = top10Attributes

fig, axes = plt.subplots(5, 2, figsize=(15, 20))
axes = axes.flatten()

for ax, col, name in zip(axes, pair_vars, pair_vars):
    sns.boxplot(data=df_observation, x="oximetry", y=col, ax=ax)
    ax.set_title(f"{name} vs oximetry")
    ax.set_xlabel("oximetry")
    ax.set_ylabel(name)

plt.tight_layout()
plt.show()

### Záver párovej analyzy

| Atribút               | `oximetry=0`      | `oximetry=1` | note                                                            |
|-----------------------|-------------------|--------------|-----------------------------------------------------------------|
| PI (Perfusion Index)  | median je nizsie  | vyssi median | signál horsie prechadza, ked je perfuzia slabá -> meranie zlyha |
| Motion/Activity index | median vyssi      | median nizsi | pohyb = nevidi ciste PPG = nedokaze zmerat                      |
| Signal Quality Index  | median nizsi      | median vyssi | ak je kvalita nizka, nevie dopocitat saturacie -> meranie zlyhá |
| Skin Temperature      | teplota je nizsia | -            | studena koza -> slaby signal                                    |
| EtCo2                 | trosku nizsie     | -            | stabilne dychanie = lepsia perfuzia = lepsi signal              |


Zvysne atribúty maju minimálne rozdiely, to znamena, ze nemajú primárny vplyv na to, ci senzor vobec nieco meria. Vyplyva nam z toho, ze pravdepodobne pojde o technicky problém a nie o chorobu/stav pacienta.


Oxymetria nezlyháva preto, že by vitálne funkcie pacienta boli patologické,
ale preto, že technické podmienky pre meranie nie sú ideálne (pohyb, nízka perfúzia, slabá kvalita PPG signálu).

Párová analýza ukázala, že atribúty súvisiace s kvalitou senzorického signálu - `Motion/Activity index`, `Perfusion Index`, `Signal Quality Index` a čiastočne aj `EtCO₂` majú oveľa výraznejší rozdiel medzi skupinami `oximetry=0` a `oximetry=1` ako fyziologické vitálne parametre (BP, RR, FiO₂, PRV). To naznačuje, že *príčinou neúspešného merania* nie je stav pacienta, ale *technicko-senzorické podmienky* (artefakty pohybu, slabá periférna perfúzia, nízka kvalita PPG signálu).


## Scatterplots

In [ ]:
pairs = [
    ("PI", "Motion/Activity index"),
    ("PI", "Signal Quality Index"),
    ("Motion/Activity index", "Signal Quality Index")
]

def corr_text(df, a, b):
    r = df[[a, b]].corr().iloc[0,1]
    return f"{a} vs {b}  (r = {r:.2f})"

#hustotné (hexbin) – veľmi prehľadné pri 12k záznamoch
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (x, y) in zip(axes, pairs):
    hb = ax.hexbin(
        df_observation[x], df_observation[y],
        gridsize=40, cmap="magma", mincnt=1
    )
    ax.set_title(corr_text(df_observation, x, y))
    ax.set_xlabel(x); ax.set_ylabel(y)
    fig.colorbar(hb, ax=ax, label="počet bodov")

plt.tight_layout()
plt.show()

- PI ↔ Motion: odhalí, či slabšia perfúzia súvisí s väčšími artefaktmi (zvyčajne veľmi slabá/žiadna lineárna väzba).
- PI ↔ SQI: ukáže, že pri nízkom PI býva nižšia kvalita signálu (očakávaná pozitívna väzba).
- Motion ↔ SQI: viac pohybu → horšia kvalita (očakávaná negatívna väzba).

Scatter grafy medzi PI, Motion/Activity indexom a Signal Quality Indexom ukazujú, že tieto atribúty spolu takmer nekorelujú lineárne (r ≈ 0). To znamená, že predstavujú nezávislé zdroje problémov pri oxymetrickom meraní: pohyb pacienta a periférna perfúzia (slabý pulzný signál). 

Signal Quality Index funguje ako výsledná metrika kvality PPG signálu, ktorá klesá pri oboch typoch problémov, no nevzniká z priamej lineárnej väzby medzi PI a Motion. 


Týmto scatter grafy potvrdzujú, že príčiny neúspešného merania sú rôzne a multifaktoriálne, čo odôvodňuje formuláciu viacerých hypotéz.


In [ ]:
df_observation['oximetry'] = df_observation['oximetry'].astype('category')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pairs = [
    ("PI", "Motion/Activity index"),
    ("PI", "Signal Quality Index"),
    ("Motion/Activity index", "Signal Quality Index")
]

for ax, (x, y) in zip(axes, pairs):
    sns.scatterplot(
        data=df_observation,
        x=x, y=y,
        hue="oximetry",
        palette={0: "darkturquoise", 1: "darkmagenta"},
        alpha=0.35, s=12,
        ax=ax
    )
    ax.set_title(f"{x} vs {y} podľa stavu oximetrie")
    ax.set_xlabel(x)
    ax.set_ylabel(y)

plt.tight_layout()
plt.show()

- fialové body = úspesne merania
- tyrkysové body = vytlacene k okrajom => nizky PI, vysoky  = failure


## Výsledky EDA
### Záverečné zhrnutie fázy 1.1 - Prieskum dát

Analýza datasetu ukázala, že namerané vitálne funkcie (SpO₂, HR, RR, EtCO₂, BP, CO
a pod.) sa pohybujú v fyziologicky stabilných rozsahoch, čo naznačuje, že dáta
pochádzajú zo stabilizovanej populácie pacientov bez výrazných hypoxických alebo
akútnych udalostí.

Predikovaná premenná `oximetry` nie je hodnota SpO₂, ale binárna informácia o
tom, či sa senzorom podarilo zmerať saturáciu (1 = úspešné meranie, 0 = zlyhanie).
To znamená, že úlohou nie je predikovať úroveň oxygenácie, ale **predikovať
dostupnosť/spoľahlivosť oximetrického merania**.

Párová analýza ukázala, že dostupnosť merania nesúvisí priamo s vitálnymi
parametrami (napr. HR, BP, CO), ale so senzormi a signálovými podmienkami.
Najsilnejšie súvislosti boli pozorované medzi:
- `oximetry` a **Motion/Activity index** (pohybové artefakty),
- `oximetry` a **Perfusion Index (PI)** (priechodnosť periférneho prietoku),
- `oximetry` a **Signal Quality Index**,
- čiastočne `oximetry` a **EtCO₂** (respiračná stabilita).

Tieto výsledky naznačujú, že dôvodom zlyhaného oximetrického merania sú najmä
technicko-senzorické podmienky (pohyb, slabá perfúzia, nízky signál), nie
fyziologická zmena vitálneho stavu pacienta.

Z týchto dôvodov bude fáza **1.2 – Čistenie dát** zameraná na:
- identifikáciu záznamov s nízkou kvalitou signálu (oximetry = 0)
- overenie, či ide o artefakty alebo reálne zhoršené meranie
- rozhodnutie, či tieto záznamy ponechať, upraviť alebo odstrániť

### Sanity check pre PI
Atribút `PI` _(Perfusion Index)_ reprezentuje intenzitu signálu (= koľko krvi ide do končatiny) a určuje, či meranie prebehne.
- Slabé prekrvenie → nízky PI → senzor „nevidí“ pulz


In [ ]:
df_observation.groupby('oximetry', observed=False)['PI'].describe()

`oximetry=1` -> senzor dokázal zmerať SpO₂ (= signál dostatočne kvalitný)

`oximetry=0` -> senzor nedokázal zmerať hodnotu (zlyhanie)

- PI je nizsi pri zlyhaní meriania (`oximetry=0`) => dáta sa správajú konzistentne s fyziológiou
- oximetry reprezentuje kvalitu dostupnosti signálu

Porovnania PI podľa oximetry=0/1 potvrdil, že diferenciácia medzi týmito dvoma stavmi nie je v hodnote saturácie, ale v podmienkach merania, čo podporuje hypotézu o tom, že oximetry reprezentuje kvalitu dostupnosti signálu.

Výsledok sanity checku ukázal, že pri `oximetry=0` sú hodnoty PI nižšie ako pri `oximetry=1`, čo naznačuje, že rozdiel medzi skupinami je fyziologicky validný a nie náhodný. Týmto sanity check potvrdzuje, že má zmysel ďalej formálne testovať hypotézu o vplyve perfúzie na úspešnosť oxymetrického merania.



# 1.2 Identifikácia problémov, integrácia a čistenie dát

- Pre pripomenutie si zobrazime aky format maju nase data

In [ ]:
df_observation.info()

In [ ]:
df_station.info()

In [ ]:
df_patient.info()

- Vidime datumy v nespravnom formate v **df_station.revision** a **df_patient.registration**
- Pretypujeme ich na datetime, tym aj zjednotime ich format

In [ ]:
df_station['revision'] = pd.to_datetime(
    df_station['revision'],
    format='mixed',
    errors='coerce'
)
df_station.info()

In [ ]:
df_patient['registration'] = pd.to_datetime(
    df_patient['registration'],
    format='mixed',
    errors='coerce'
)
df_patient['birthdate'] = pd.to_datetime(
    df_patient['birthdate'],
    format='mixed',
    errors='coerce'
)
df_patient.info()

- vidime ze current_location je object ktory vyzera takto:

In [ ]:
df_patient[['current_location']].head()

- stlpec current_location rozbijeme na dva dalsie stlpce -> latitude a longitude

In [ ]:
def extractCoords(value):
    if pd.isna(value):
        return pd.NA, pd.NA

    cleaned = (value
               .replace("Decimal", "")
               .replace("(", "")
               .replace(")", ""))
    lat_str, lon_str = cleaned.split(",")

    lat = float(lat_str.replace("'", "").strip())
    lon = float(lon_str.replace("'", "").strip())

    return lat, lon

In [ ]:
df_patient['latitude'], df_patient['longitude'] = zip(*df_patient['current_location'].apply(extractCoords))
df_patient[['current_location', 'latitude', 'longitude']].head()

- current_location nam uz netreba tak ho vymazeme

In [ ]:
df_patient = df_patient.drop(columns=['current_location'])

### Skontrolujeme duplikaty

In [ ]:
print(f"[i] Observation dupes: {df_observation.duplicated().sum()}")
print(f"[i] Station dupes:     {df_station.duplicated().sum()}")
print(f"[i] Patient dupes:     {df_patient.duplicated().sum()}")

- v datach nemame ziadne duplikaty, yay :3

### Vyriesime chybajuce hodnoty

In [ ]:
df_observation.isnull().sum()

- v df_observation niesu ziadne chybajuce data

In [ ]:
df_station.isnull().sum()

In [ ]:
df_station[df_station['code'].isna()]

- vidime ze chybaju kody krajiny pre data z Afriky z mesta Windhoek

In [ ]:
df_station[df_station['location'] == 'Africa/Windhoek']

- ziadne ine zaznami z mesta Windhoek ani nemame
- najdeme najviac vyskytovany kod (`mode`) v celej afrike, ten pouzijeme na nahradenie pre nase 2 zaznamy

In [ ]:
df_station[df_station['location'].str.split("/").str[0] == 'Africa']

In [ ]:
country_code = df_station[df_station['location'].str.split("/").str[0] == 'Africa']['code'].mode().iat[0]
country_code

In [ ]:
df_station['code'] = df_station['code'].fillna(country_code)

In [ ]:
df_station.isnull().sum()

In [ ]:
df_patient.isnull().sum()

- tomto DataFrame mame vela chybajucich data, plan je nasledovny:

| Column    | Missing out of 2069 | % Missing | Solution                                |
|-----------|:-------------------:|:---------:|-----------------------------------------|
| residence |        2069         |   100%    | Vymazeme                                |
| birthdate |         931         |  44.998%  | nahradime s mean()                      |
| job       |        1448         |  69.986%  | Vymazeme (nemal by byt vyznamny pre ML) |
| latitude  |         103         |  4.978%   | nahradime s mean()                      |
| longitude |         103         |  4.978%   | nahradime s mean()                      |

In [ ]:
df_patient = df_patient.drop(columns=['residence', 'job'])
df_patient.info()

In [ ]:
df_patient['birthdate'] = df_patient['birthdate'].fillna(df_patient['birthdate'].mean())
df_patient['latitude'] = df_patient['latitude'].fillna(df_patient['latitude'].mean())
df_patient['longitude'] = df_patient['longitude'].fillna(df_patient['longitude'].mean())

df_patient.isnull().sum()

In [ ]:
df_patient.info()

In [ ]:
df_patient.head(10)

### Upraceme outliers

- nahodnym vyberom som zvolil stlpec $EtCO_2$ ako obet pre riesenie problemu outlierov **mazanim** a stastnou nahodou ich je velmi malo, takze neprideme o vela dat

In [ ]:
sns.boxplot(y=df_observation['EtCO₂'])
plt.title('Boxplot of Observation.EtCO₂')
plt.ylabel('EtCO₂')
plt.show()

- zopar pomocnych funkcii

In [ ]:
def getOutliers(a):
    lower = a.quantile(0.25) - 1.5 * stats.iqr(a)
    upper = a.quantile(0.75) + 1.5 * stats.iqr(a)

    return (a > upper) | (a < lower)

In [ ]:
def cleanOutliers(data, cols):
    for c in cols:
        mask = getOutliers(data[c])

        below = data[c].quantile(0.05)
        above = data[c].quantile(0.95)

        for i in data.index[mask]:
            value = data.at[i, c]

            if value < below:
                data.at[i, c] = below
            elif value > above:
                data.at[i, c] = above


In [ ]:
def histogramiada(df, columns, bins=30):

    fig, axes = plt.subplots(3, 3, figsize=(12, 10))
    axes = axes.flatten()

    for i, col in enumerate(columns[:9]):
        sns.histplot(df[col], bins=bins, kde=True, ax=axes[i], color=sns.color_palette("tab10")[i % 10])
        axes[i].set_title(col, fontsize=11)
        axes[i].set_xlabel("")
        axes[i].set_ylabel("")

    plt.tight_layout()
    plt.show()

- `getOutliers()` vracia zoznam true/false hodnot, cize vlastne masku ktora hovori ci zaznam je alebo nieje outlier
- pomocou nej ich vyfiltrujeme prec

In [ ]:
etco2_mask = getOutliers(df_observation['EtCO₂'])
df_observation = df_observation[~etco2_mask] # ~ inverts it -> keeps only rows where it’s not an outlier
etco2_mask

# odstranime tento stlpec z listu, aby sme ten list neskor mohli pouzit ako vstup do funkcie pre cistenie outlierov zvysnych zvolenych atributov
top10Attributes.remove('EtCO₂')

In [ ]:
sns.boxplot(y=df_observation['EtCO₂'])
plt.title('Boxplot of Observation.EtCO₂')
plt.ylabel('EtCO₂')
plt.show()

- pomocou `histogramiada()` si zobrazim distribucie aby som ich vedel porovnat pred a po cisteni

In [ ]:
histogramiada(df_observation, top10Attributes)

- zvysne atributy precistime od outlierov tym ze ich hodnotu zmenime na hranicne hodnoty rozdelenia

In [ ]:
cleanOutliers(df_observation, top10Attributes)

In [ ]:
histogramiada(df_observation, top10Attributes)

# 1.3 Hypotézy
- prediokovaná premenná je **oximtery** (0/1)
- najviac súvisiacimi atribútmi sú **Motion/Activity index**, **PI**, **Signal Quality Index**, **EtCO₂**

### Hypotéza - vyplyv pohybu na meranie Motion/Activity index
Pri neúspešných meraniach je vyšší podiel artefaktov spôsobených pohybom → senzor nedokáže stabilizovať PPG signál → oxymetria zlyhá.

$H_0$: Motion/activity index nie je rozdielny medzi `oximetry=0` a `oximetry=1`.

$H_1$: Motion/activity index je vyssi pri neuspesnom merani (`oximetry=0`) ako pri uspesnom (`oximetry=1`).

### Hypotéza - vyplyv perfúzie (PI)
Nízka periférna perfúzia → slabší PPG signál → snímač síce „vidí prítomnosť prsta“, ale nemá dostatok amplitúdy na spoľahlivý výpočet SpO₂.


$H_0$: Perfusion Index (PI) nie je rozdielny medzi `oximetry=0` a `oximetry=1`

$H_1$: Perfusion Index je nizsi pri neuspesnom merani (`oximtry=0`) ako pri uspesnom (`oximetry=1`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# H1 - Motion/Activity index vs oximetry
sns.boxplot(
    data=df_observation,
    x="oximetry",
    y="Motion/Activity index",
    ax=axes[0]
)
axes[0].set_title("H1: Motion index vs oximetry")
axes[0].set_xlabel("oximetry (0=fail, 1=success)")
axes[0].set_ylabel("Motion/Activity index")

# H2 - PI vs oximetry
sns.boxplot(
    data=df_observation,
    x="oximetry",
    y="PI",
    ax=axes[1]
)
axes[1].set_title("H2: PI vs oximetry")
axes[1].set_xlabel("oximetry (0=fail, 1=success)")
axes[1].set_ylabel("PI")

plt.tight_layout()
plt.show()

In [ ]:
SIGNIFICANCE_LEVEL = 0.05

- T-test pre Motion/Activity index

In [ ]:
sample_0 = df_observation[df_observation['oximetry'] == 0]['Motion/Activity index']
sample_1 = df_observation[df_observation['oximetry'] == 1]['Motion/Activity index']

t_stat, p_value = stats.ttest_ind(sample_0, sample_1)

print(f'[i] P-value: {p_value}')

if p_value <= SIGNIFICANCE_LEVEL:
    print('[-] Odmietame H_0')
    print('[+] Prijmame H_1')
    print('[!] Motion/activity index je vyssi pri neuspesnom merani (oximetry=0) ako pri uspesnom (oximetry=1)')
else:
    print('[-] Odmietame H_1')
    print('[+] Prijmame H_0')
    print('[!] Motion/activity index nie je rozdielny medzi (oximetry=0) a (oximetry=1)')


- overime statisticku silu nasho testu

In [ ]:
effect_size = (sample_0.mean() - sample_1.mean()) / np.sqrt(((sample_0.std()**2 + sample_1.std()**2) / 2))
analysis = TTestIndPower()
power = analysis.solve_power(effect_size=effect_size, nobs1=len(sample_0), alpha=0.05, ratio=len(sample_1)/len(sample_0))

print(f'[i] Statisticka sila testu: {power}')

if power < 0.5:
    print("[!] Mala statisticka sila")
elif power < 0.8:
    print("[~] Ciastocna statisticka sila, ale nemusi byt dostatok dat")
else:
    print("[+] Silna statisticka sila")

- T-test pre PI

In [ ]:
sample_0 = df_observation[df_observation['oximetry'] == 0]['PI']
sample_1 = df_observation[df_observation['oximetry'] == 1]['PI']

t_stat, p_value = stats.ttest_ind(sample_0, sample_1)

print(f'[i] P-value: {p_value}')

if p_value <= SIGNIFICANCE_LEVEL:
    print('[-] Odmietame H_0')
    print('[+] Prijmame H_1')
    print('[!] Perfusion Index je nizsi pri neuspesnom merani (oximtry=0) ako pri uspesnom (oximetry=1)')
else:
    print('[-] Odmietame H_1')
    print('[+] Prijmame H_0')
    print('[!] Perfusion Index (PI) nie je rozdielny medzi (oximetry=0) a (oximetry=1)')

- overime statisticku silu nasho testu

In [ ]:
effect_size = (sample_0.mean() - sample_1.mean()) / np.sqrt(((sample_0.std()**2 + sample_1.std()**2) / 2))
analysis = TTestIndPower()
power = analysis.solve_power(effect_size=effect_size, nobs1=len(sample_0), alpha=0.05, ratio=len(sample_1)/len(sample_0))

print(f'[+] Statisticka sila testu: {power}')

if power < 0.5:
    print("[!] Mala statisticka sila")
elif power < 0.8:
    print("[~] Ciastocna statisticka sila, ale nemusi byt dostatok dat")
else:
    print("[+] Silna statisticka sila")